# Text Preprocessing - Amazon Recommender System

This notebook prepares textual features for recommendation models.

We build two pipelines:

1. TF-IDF preprocessing (cleaned text)
2. BERT preprocessing (minimal cleaning)

Goal:
Convert raw textual data into meaningful representations for modeling.

In [1]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
# Download required resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/souravsinha/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/souravsinha/nltk_data...


True

## Load Cleaned Dataset

We use the cleaned dataset prepared in the previous notebook.

In [3]:
DATA_PROCESSED = "../data/processed/"

df = pd.read_csv(DATA_PROCESSED + "cleaned_data.csv")

df.head()

,reviewerID,asin,overall,reviewText,summary,reviewTime,title,brand,category_list,category_text,description
0,A3NHUQ33CFH3VM,1118461304,5.0,Not one thing in this book seemed an obvious o...,Clear on what leads to innovation,2013-11-27,NaN,NaN,[],unknown,unknown
1,A3SK6VNBQDNBJE,1118461304,5.0,I have enjoyed Dr. Alan Gregerman's weekly blo...,Becoming more innovative by opening yourself t...,2013-11-01,NaN,NaN,[],unknown,unknown
2,A3SOFHUR27FO3K,1118461304,5.0,Alan Gregerman believes that innovation comes ...,The World from Different Perspectives,2013-10-10,NaN,NaN,[],unknown,unknown
3,A1HOG1PYCAE157,1118461304,5.0,"Alan Gregerman is a smart, funny, entertaining...",Strangers are Your New Best Friends,2013-10-09,NaN,NaN,[],unknown,unknown
4,A26JGAM6GZMM4V,1118461304,5.0,"As I began to read this book, I was again remi...","How and why it is imperative to engage, learn ...",2013-09-07,NaN,NaN,[],unknown,unknown


## Combine Text Features

We combine multiple textual fields into a single column.

This improves recommendation quality by capturing:
- Product title
- Review summary
- Full review text
- Category
- Description

In [4]:
df["combined_text"] = (
    df["title"].fillna('') + " " +
    df["summary"].fillna('') + " " +
    df["reviewText"].fillna('') + " " +
    df["category_text"].fillna('') + " " +
    df["description"].fillna('')
)

df[["combined_text"]].head()

,combined_text
0,Clear on what leads to innovation Not one thi...
1,Becoming more innovative by opening yourself ...
2,The World from Different Perspectives Alan Gr...
3,Strangers are Your New Best Friends Alan Greg...
4,"How and why it is imperative to engage, learn..."


## Text Cleaning for TF-IDF

Steps:
- Lowercasing
- Remove HTML
- Remove special characters
- Remove extra spaces
- Tokenization
- Remove stopwords
- Lemmatization

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if pd.isna(text):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # Remove special characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords + lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

## Apply Cleaning (TF-IDF Pipeline)

In [6]:
from tqdm import tqdm

tqdm.pandas()

df["clean_text"] = df["combined_text"].progress_apply(clean_text)

df[["combined_text", "clean_text"]].head()

100%|██████████| 590985/590985 [00:31<00:00, 18808.28it/s]


,combined_text,clean_text
0,Clear on what leads to innovation Not one thi...,clear lead innovation one thing book seemed ob...
1,Becoming more innovative by opening yourself ...,becoming innovative opening conversation stran...
2,The World from Different Perspectives Alan Gr...,world different perspective alan gregerman bel...
3,Strangers are Your New Best Friends Alan Greg...,stranger new best friend alan gregerman smart ...
4,"How and why it is imperative to engage, learn...",imperative engage learn collaborate stranger b...


## Prepare Text for BERT

BERT works best with minimal preprocessing.

We only:
- Lowercase
- Remove extra spaces

In [7]:
def clean_text_bert(text):
    if pd.isna(text):
        return ""
    
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df["bert_text"] = df["combined_text"].apply(clean_text_bert)

df[["bert_text"]].head()

,bert_text
0,clear on what leads to innovation not one thin...
1,becoming more innovative by opening yourself t...
2,the world from different perspectives alan gre...
3,strangers are your new best friends alan grege...
4,"how and why it is imperative to engage, learn ..."


## Save Preprocessed Dataset

We store processed text for modeling.

In [8]:
df.to_csv(DATA_PROCESSED + "text_processed.csv", index=False)

In [9]:
df.head()

,reviewerID,asin,overall,reviewText,summary,reviewTime,title,brand,category_list,category_text,description,combined_text,clean_text,bert_text
0,A3NHUQ33CFH3VM,1118461304,5.0,Not one thing in this book seemed an obvious o...,Clear on what leads to innovation,2013-11-27,NaN,NaN,[],unknown,unknown,Clear on what leads to innovation Not one thi...,clear lead innovation one thing book seemed ob...,clear on what leads to innovation not one thin...
1,A3SK6VNBQDNBJE,1118461304,5.0,I have enjoyed Dr. Alan Gregerman's weekly blo...,Becoming more innovative by opening yourself t...,2013-11-01,NaN,NaN,[],unknown,unknown,Becoming more innovative by opening yourself ...,becoming innovative opening conversation stran...,becoming more innovative by opening yourself t...
2,A3SOFHUR27FO3K,1118461304,5.0,Alan Gregerman believes that innovation comes ...,The World from Different Perspectives,2013-10-10,NaN,NaN,[],unknown,unknown,The World from Different Perspectives Alan Gr...,world different perspective alan gregerman bel...,the world from different perspectives alan gre...
3,A1HOG1PYCAE157,1118461304,5.0,"Alan Gregerman is a smart, funny, entertaining...",Strangers are Your New Best Friends,2013-10-09,NaN,NaN,[],unknown,unknown,Strangers are Your New Best Friends Alan Greg...,stranger new best friend alan gregerman smart ...,strangers are your new best friends alan grege...
4,A26JGAM6GZMM4V,1118461304,5.0,"As I began to read this book, I was again remi...","How and why it is imperative to engage, learn ...",2013-09-07,NaN,NaN,[],unknown,unknown,"How and why it is imperative to engage, learn...",imperative engage learn collaborate stranger b...,"how and why it is imperative to engage, learn ..."
